### Data Ingestion

In [1]:
from langchain_community.document_loaders import PyMuPDFLoader
from pathlib import Path

def load_and_split_documents():
    all_doc=[]
    directory_path=Path("../data/pdf_files")  # Specify the directory containing your PDF files

    files=list(directory_path.glob('*.pdf'))
    print(f"Found {len(files)} PDF files in the directory.")
    
    # Load documents
    for pdf in files:
        print(f"Loading document: {pdf.name}")
        try:
            loader = PyMuPDFLoader(str(pdf))
            documents = loader.load()
            for doc in documents:
                doc.metadata["source_file"] = pdf.name
                doc.metadata['file_type']="pdf"

            all_doc.extend(documents)

        except Exception as e:
            print(f"Error loading {pdf.name}: {e}")
    print(f"Total documents loaded: {len(all_doc)}")
    return all_doc
    

documents=load_and_split_documents()

/home/resumetozero/Documents/Projects/BTP/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Found 1 PDF files in the directory.
Loading document: Sql_1739036725.pdf
Total documents loaded: 9


In [7]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=800, chunk_overlap=100):
    all_chunks = []

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, 
        chunk_overlap=chunk_overlap, 
        separators=["\n\n","\n"," ",". ","","; "],
        length_function=len)
    
    split_docs=text_splitter.split_documents(documents)
    print(f"Split {len(documents)} Documents into {len(split_docs)} chunks.")

    if split_docs:
        all_chunks.extend(split_docs)
    return all_chunks

split_docs(documents)

Split 9 Documents into 9 chunks.


[Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/Sql_1739036725.pdf', 'file_path': '../data/pdf_files/Sql_1739036725.pdf', 'total_pages': 9, 'format': 'PDF 1.4', 'title': '', 'author': '', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0, 'source_file': 'Sql_1739036725.pdf', 'file_type': 'pdf'}, page_content="When you code has different WHERE conditions,\n having a WHERE 1=1 simplifies the logic. \nThis is because 1=1 is always true and doesn't affect\n the query's actual result, but it allows you to mor\ne easily append additional conditions without needing \nspecial handling for the first condition.\nIt improves the overall readability of the code as well.\n1. WHERE 1=1"),
 Document(metadata={'producer': '', 'creator': '', 'creationdate': '', 'source': '../data/pdf_files/Sql_1739036725.pdf', 'file_path': '../data/pdf_files/Sql_1739036725.pdf', 'total_pages': 9, 'format': 'PDF 1

In [8]:
import numpy as np
from sentence_transformers import SentenceTransformer
import chromadb
from chromadb.config import Settings
import uuid
from sklearn.metrics.pairwise import cosine_similarity


In [11]:

class EmbeddingManager:
    def __init__(self, embedding_model="BAAI/bge-base-en-v1.5"):   #emilyalsentzer/Bio_ClinicalBERT
        self.embedding_model = embedding_model
        self.model=None
        self._load_model_()
    
    def _load_model_(self):
        print("Loading embedding model...")
        self.model = SentenceTransformer(self.embedding_model)
        print(f"Embedding model loaded successfully. Dimensionality: {self.model.get_sentence_embedding_dimension()}")

    def generate_embeddings(self, texts)->np.ndarray:
        if not self.model:
            raise ValueError("Embedding model is not loaded.")
        print(f"Generating embeddings for {len(texts)} texts...")
        embeddings = self.model.encode(texts, show_progress_bar=True, normalize_embeddings=True)
        print("Embeddings generated successfully.")
        return embeddings
    
##initialize embedding manager

embedding_manager = EmbeddingManager()
embedding_manager


Loading embedding model...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 8421.02it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embedding model loaded successfully. Dimensionality: 768


In [ ]:
import os

class vectorDBManager:
    def __init__(self, collection_name="pdf_documents", persist_directory="data/vectordb_storage"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.client = None
        self.collection = None
        self._initialize_chromadb_()
    
    def _initialize_chromadb(self):
        try:
            print("Initializing ChromaDB client...")
            os.makedirs(self.persist_directory, exist_ok=True)
            self.client = chromadb.PersistentClient(Path=self.persist_directory)

            self.collection = self.client.get_or_create_collection(
                name=self.collection_name, 
                metadata={"description": "pdf document for RAG system"})
            
            print(f"ChromaDB client initialized successfully. Collection '{self.collection_name}' is ready.")
            print(f"Existing collections: {self.collection._count()}")
        except Exception as e:
            print(f"Error initializing ChromaDB client: {e}")  
    

    def add_document(self, document, embedding: np.ndarray):
        if len(embedding)!=len(document):
            raise ValueError("Embedding length does not match document length.")
        print(f"Adding {len(document)} documents to the collection '{self.collection_name}'...")
        ids=[]
        metadatas=[]
        document_texts=[]
        embeddings_list=[]
        
        for i, (doc, emb) in enumerate(zip(document, embedding)):
            doc_id = str(uuid.uuid4())
            ids.append(doc_id)
            metadatas.append(doc.metadata)
            document_texts.append(doc.page_content)
            embeddings_list.append(emb.tolist())